# Phase 5: Gradio Integration

In this phase, we create a user-friendly web interface for uploading a skin image, asking an optional question, uploading an optional document, and displaying the model prediction and structured medical report.

**Check Gradio Availability**

In [1]:
import importlib.util

status = (
    "Installed"
    if importlib.util.find_spec("gradio")
    else "Not installed"
)

print("Gradio:", status)

Gradio: Installed


In [2]:
%pip install gradio

Note: you may need to restart the kernel to use updated packages.


In [3]:
import gradio as gr

print("Gradio version:", gr.__version__)

/home/arslan/projects/skin-lesion-classifier/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Gradio version: 6.19.0


**Import Libraries**

In [7]:
import os
import json
import numpy as np
import faiss
import tensorflow as tf

from sentence_transformers import SentenceTransformer
from tensorflow.keras.utils import load_img, img_to_array
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from pypdf import PdfReader
from docx import Document

**Set Project And Backend File Paths**

In [8]:
project_path = "/home/arslan/projects/skin-lesion-classifier"

model_path = (
    project_path
    + "/models/mobilenet/mobilenetv2_model.keras"
)

knowledge_path = (
    project_path
    + "/rag_data/documents/skin_lesion_knowledge.json"
)

faiss_index_path = (
    project_path
    + "/rag_data/index/skin_lesion_faiss.index"
)

print("Paths created successfully.")

Paths created successfully.


**Verify Backend Files**

In [9]:
print("Model exists:", os.path.exists(model_path))
print("Knowledge base exists:", os.path.exists(knowledge_path))
print("FAISS index exists:", os.path.exists(faiss_index_path))

Model exists: True
Knowledge base exists: True
FAISS index exists: True


**Load Medical Knowledge Base And FAISS Index**

In [10]:
with open(
    knowledge_path,
    "r",
    encoding="utf-8"
) as file:
    knowledge_base = json.load(file)

faiss_index = faiss.read_index(
    faiss_index_path
)

print("Knowledge entries:", len(knowledge_base))
print("FAISS vectors:", faiss_index.ntotal)

Knowledge entries: 7
FAISS vectors: 7


**Load Sentence Transformer Model**

In [11]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1351.58it/s]


Embedding model loaded successfully.


**Load Frozen MobileNetV2 Model**

In [12]:
mobilenet_model = tf.keras.models.load_model(
    model_path,
    compile=False
)

print("MobileNetV2 model loaded successfully.")

W0000 00:00:1781994052.528796    4328 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1781994052.533197    4328 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1781994052.537117    4328 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5129 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5060 Laptop GPU, pci bus id: 0000:02:00.0, compute capability: 12.0a


MobileNetV2 model loaded successfully.


**Define Lesion Class Names**

In [13]:
class_codes = [
    "akiec",
    "bcc",
    "bkl",
    "df",
    "mel",
    "nv",
    "vasc"
]

class_names = {
    "akiec": "Actinic Keratoses and Intraepithelial Carcinoma",
    "bcc": "Basal Cell Carcinoma",
    "bkl": "Benign Keratosis-like Lesions",
    "df": "Dermatofibroma",
    "mel": "Melanoma",
    "nv": "Melanocytic Nevi",
    "vasc": "Vascular Lesions"
}

**Image Preprocessing Function**

In [14]:
IMAGE_SIZE = (224, 224)

def prepare_image(image_path):
    image = load_img(
        image_path,
        target_size=IMAGE_SIZE
    )

    image = img_to_array(image)
    image = np.expand_dims(image, axis=0)
    image = preprocess_input(image)

    return image

**Top-3 Prediction Function**

In [15]:
def predict_top3(image_path):
    image = prepare_image(image_path)

    with tf.device("/CPU:0"):
        probabilities = mobilenet_model(
            image,
            training=False
        ).numpy()[0]

    top_indices = np.argsort(
        probabilities
    )[-3:][::-1]

    top_predictions = []

    for index in top_indices:
        code = class_codes[index]

        top_predictions.append({
            "class_code": code,
            "class_name": class_names[code],
            "probability": float(probabilities[index])
        })

    return top_predictions

**Medical Knowledge Retrieval Function**

In [16]:
def retrieve_knowledge(query, top_k=1):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = faiss_index.search(
        query_embedding,
        k=top_k
    )

    results = []

    for score, index in zip(scores[0], indices[0]):
        result = knowledge_base[index].copy()
        result["similarity_score"] = float(score)
        results.append(result)

    return results

**Document Text Extraction Function**

In [17]:
def extract_document_text(file_path):
    lower_path = file_path.lower()

    if lower_path.endswith(".pdf"):
        reader = PdfReader(file_path)

        text = "\n".join(
            page.extract_text() or ""
            for page in reader.pages
        )

    elif lower_path.endswith(".docx"):
        document = Document(file_path)

        text = "\n".join(
            paragraph.text
            for paragraph in document.paragraphs
        )

    elif lower_path.endswith(".txt"):
        with open(file_path, "r", encoding="utf-8") as file:
            text = file.read()

    else:
        raise ValueError(
            "Only PDF, DOCX and TXT files are supported."
        )

    if not text.strip():
        raise ValueError(
            "No readable text was found in the document."
        )

    return text

**Document Chunking Function**

In [18]:
def split_text_into_chunks(text, chunk_size=500):
    words = text.split()
    chunks = []

    for start in range(0, len(words), chunk_size):
        chunk = " ".join(
            words[start:start + chunk_size]
        )

        chunks.append(chunk)

    return chunks

**Uploaded Document FAISS Index**

In [19]:
def build_document_index(document_path):
    text = extract_document_text(document_path)

    document_chunks = split_text_into_chunks(text)

    document_embeddings = embedding_model.encode(
        document_chunks,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    document_index = faiss.IndexFlatIP(
        document_embeddings.shape[1]
    )

    document_index.add(document_embeddings)

    return document_chunks, document_index

**Uploaded Document Retrieval Function**

In [20]:
def retrieve_from_document(
    query,
    document_chunks,
    document_index,
    top_k=3
):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = document_index.search(
        query_embedding,
        k=min(top_k, len(document_chunks))
    )

    results = []

    for score, index in zip(scores[0], indices[0]):
        results.append({
            "text": document_chunks[index],
            "similarity_score": float(score)
        })

    return results

**Structured Medical Report Function**

In [21]:
def generate_extended_medical_report(
    top_predictions,
    retrieved_information,
    user_question="",
    document_results=None
):
    report = "# Skin Lesion Analysis Report\n\n"

    report += "## Top-3 Predictions\n"
    for prediction in top_predictions:
        report += (
            f"- **{prediction['class_name']}**: "
            f"{prediction['probability'] * 100:.2f}%\n"
        )

    report += "\n## Medical Information\n"
    report += f"**Description:** {retrieved_information['description']}\n\n"
    report += f"**Typical Signs:** {retrieved_information['typical_signs']}\n\n"
    report += f"**Risk Level:** {retrieved_information['risk_level']}\n\n"
    report += f"**Recommended Action:** {retrieved_information['recommended_action']}\n\n"
    report += f"**Source:** {retrieved_information['source']}\n"

    if user_question.strip():
        report += f"\n## User Question\n{user_question}\n"

    if document_results:
        report += "\n## Uploaded Document Information\n"
        for result in document_results:
            report += f"- {result['text']}\n"

    report += (
        "\n## Medical Disclaimer\n"
        "This system is for educational purposes only and does not replace "
        "professional diagnosis or advice from a dermatologist."
    )

    return report

**Complete Backend Pipeline Function**

In [22]:
def run_pipeline_with_document(
    image_path,
    user_question="",
    document_path=None
):
    top_predictions = predict_top3(image_path)
    predicted_class = top_predictions[0]["class_name"]

    query = (
        f"Medical information, typical signs, risk level, "
        f"and recommended action for {predicted_class}. "
        f"{user_question}"
    )

    retrieved_information = retrieve_knowledge(
        query,
        top_k=1
    )[0]

    document_results = []

    if document_path:
        chunks, document_index = build_document_index(
            document_path
        )

        document_results = retrieve_from_document(
            f"{predicted_class}. {user_question}",
            chunks,
            document_index,
            top_k=3
        )

    return generate_extended_medical_report(
        top_predictions,
        retrieved_information,
        user_question,
        document_results
    )

**Gradio Analysis Function**

**Gradio Interface**

In [23]:
def analyze_lesion(image_path, user_question, document_path):
    if not image_path:
        return (
            "No prediction",
            {},
            "Please upload a skin-lesion image."
        )

    try:
        top_predictions = predict_top3(image_path)

        report = run_pipeline_with_document(
            image_path,
            user_question or "",
            document_path
        )

        primary_prediction = top_predictions[0]["class_name"]

        probability_labels = {
            prediction["class_name"]: prediction["probability"]
            for prediction in top_predictions
        }

        return (
            primary_prediction,
            probability_labels,
            report
        )

    except Exception as error:
        return (
            "Analysis failed",
            {},
            f"## Error\n{error}"
        )

In [41]:
custom_css = """
.gradio-container {
    max-width: 1180px !important;
    margin: auto !important;
    padding: 35px 24px !important;
    font-family: Inter, Arial, sans-serif !important;
    background:
        radial-gradient(circle at top left, #172554, transparent 35%),
        radial-gradient(circle at top right, #164e63, transparent 30%),
        #080d18 !important;
}

#hero-section {
    text-align: center;
    padding: 42px 28px;
    margin-bottom: 32px;
    border-radius: 24px;
    background: linear-gradient(135deg, #172554, #164e63);
    border: 1px solid rgba(103, 232, 249, 0.25);
    box-shadow: 0 18px 45px rgba(0, 0, 0, 0.35);
}

#hero-section h1 {
    font-size: 40px;
    font-weight: 800;
    margin-bottom: 14px;
}

#hero-section p {
    font-size: 16px;
    color: #dbeafe;
    line-height: 1.7;
}

#input-card,
#result-card,
#report-card {
    padding: 24px;
    border-radius: 22px;
    background: rgba(15, 23, 42, 0.88);
    border: 1px solid rgba(148, 163, 184, 0.25);
    box-shadow: 0 14px 35px rgba(0, 0, 0, 0.28);
}

#input-card:hover,
#result-card:hover,
#report-card:hover {
    border-color: rgba(34, 211, 238, 0.55);
    transform: translateY(-2px);
    transition: 0.25s ease;
}

#analyze-button {
    min-height: 52px;
    font-size: 17px;
    font-weight: 700;
    border: none !important;
    border-radius: 14px !important;
    background: linear-gradient(90deg, #06b6d4, #2563eb) !important;
    box-shadow: 0 10px 25px rgba(37, 99, 235, 0.35);
}

#analyze-button:hover {
    transform: translateY(-2px);
    box-shadow: 0 14px 30px rgba(37, 99, 235, 0.48);
}

textarea,
input {
    border-radius: 12px !important;
}

#report-card {
    margin-top: 26px;
    line-height: 1.7;
}

@media (max-width: 768px) {
    .gradio-container {
        padding: 18px 12px !important;
    }

    #hero-section h1 {
        font-size: 30px;
    }
}
"""

In [42]:
with gr.Blocks(css=custom_css) as app:
    gr.Markdown(
        """
        # 🩺 AI Skin Lesion Analysis System

        Upload a skin-lesion image to receive the top predictions,  
        educational medical information, and optional document-based retrieval.

        **This system is for educational purposes only.**
        """,
        elem_id="hero-section"
    )

    with gr.Row():
        with gr.Column(scale=1, elem_id="input-card"):
            gr.Markdown("## Upload And Analyze")

            image_input = gr.Image(
                type="filepath",
                label="Skin Lesion Image",
                height=350
            )

            question_input = gr.Textbox(
                label="Optional Question",
                placeholder="Example: What is the risk level?"
            )

            document_input = gr.File(
                type="filepath",
                file_types=[".pdf", ".docx", ".txt"],
                label="Optional Medical Document"
            )

            analyze_button = gr.Button(
                "🔍 Analyze Skin Lesion",
                variant="primary",
                elem_id="analyze-button"
            )

        with gr.Column(scale=1, elem_id="result-card"):
            gr.Markdown("## Prediction Results")

            primary_output = gr.Textbox(
                label="Primary Predicted Lesion",
                interactive=False
            )

            probability_output = gr.Label(
                label="Top-3 Probabilities",
                num_top_classes=3
            )

    with gr.Column(elem_id="report-card"):
        gr.Markdown("## Educational Medical Report")

        report_output = gr.Markdown(
            "Upload an image and click **Analyze Skin Lesion** to view the report."
        )

    analyze_button.click(
        fn=analyze_lesion,
        inputs=[
            image_input,
            question_input,
            document_input
        ],
        outputs=[
            primary_output,
            probability_output,
            report_output
        ]
    )

/tmp/ipykernel_4328/2761095410.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=custom_css) as app:


In [44]:
app.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [43]:
app.close()